# Risk Parameters of Truflation EV Fiat Market

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
pd.options.plotting.backend = "plotly"

import funding
import impact
import liquidations as liq
import pricedrift as drift
import pystable
from tqdm import tqdm

Set data-specific parameters:

In [14]:
file_name = "/Users/fredericoteixeira/Projects/overlay/data/btc_ev_v12_fiat.csv"
periodicity = funding.NS[0]  # converts 1 day in seconds
cap = 10  # cap on pay off, set by governance

## Understanding the Data

Load and plot the data:

In [44]:
df = pd.read_csv(file_name).set_index("date").drop("created_at", axis=1)

df["twap"] = df["btc_ev_v12_fiat_index"].rolling(2).mean()

df.dropna(inplace=True)

df.plot()

## Funding Rate - `k`

Compute `k`, the funding related risk metric.

It means that with this `k`, if OI exists only on one side (the worst case scenario), then in $1-\alpha$% of cases, the OI will get drawn down to zero just due to funding over the course of the next `n` days.

In [51]:
today = df.index[-1]
years_ago = df.index[-2*365]

In [55]:
ks, dst = funding.generic_get_ks(
    df.loc[years_ago:today, "btc_ev_v12_fiat_index"].to_numpy(), periodicity
)

df_ks = pd.DataFrame(
    data=ks,
    columns=[alpha for alpha in funding.ALPHAS],
    index=[n/funding.NS[0] for n in funding.NS]
)
df_ks.columns.name = "alpha"
df_ks.index.name = "days"

df_ks.plot()

The result, when multipled by `1e18`, yields the amount that needs to be sent to the smart contract.

In [56]:
df_ks.loc[30, :] * 1e18

alpha
0.010    1.117478e+12
0.025    3.556678e+11
0.050    1.423950e+11
0.075    9.663572e+10
0.100    8.177599e+10
Name: 30.0, dtype: float64

## Impact - $\lambda$

## Impact - $\delta$

## Liquidations - `maintenanceMarginFraction`

## Liquidations - `maintenanceMarginBurnRate`

## Price drift - `priceDriftUpperLimit`